# 🦙 Fine-Tuning Llama 3.2 1B with ORPO for Local Deployment with Ollama

## What You'll Learn

This notebook walks you through the complete process of:

1. **Fine-tuning a Large Language Model (LLM)** using ORPO (Odds Ratio Preference Optimization)
2. **Exporting the model to GGUF format** for efficient local inference
3. **Deploying your custom model with Ollama** on your local machine

### Prerequisites
- Basic understanding of Python and machine learning concepts
- A GPU with at least 8GB VRAM (e.g., RTX 3060, RTX 4060)
- Google Colab (free tier works!) or a local CUDA-capable machine

### Why This Matters
Fine-tuning allows you to adapt a pre-trained model to your specific use case, whether it's customer support, code generation, or domain-specific knowledge. By exporting to GGUF and using Ollama, you can run your custom model **completely offline** on consumer hardware.

---

## Step 1: Environment Setup 🔧

Before we begin, we need to install the required libraries:

- **Unsloth**: A library that dramatically speeds up LLM fine-tuning (2-5x faster) while reducing memory usage by 60-80%
- **TRL (Transformer Reinforcement Learning)**: Hugging Face's library for training LLMs with preference optimization methods like ORPO, DPO, and RLHF
- **Datasets**: For loading and preprocessing training data

> ⏱️ **Note**: The first installation of Unsloth can take 5-15 minutes as it compiles CUDA kernels optimized for your GPU.

In [ ]:
# Step 1: Install Dependencies
# We use !pip to see real-time progress (Unsloth may take 5-15 min the first time)

try:
    import google.colab
    PKG = "unsloth[colab-new]"  # Colab-specific package with optimizations
except ImportError:
    PKG = "unsloth"  # Standard package for local installations

print("Installing dependencies (this may take several minutes the first time)...\n")
!pip install "{PKG}" datasets trl

from huggingface_hub import list_models
print("\n✓ Dependencies installed successfully!")

Instalando dependencias (puede tardar varios minutos la primera vez)...


✓ Dependencias instaladas.


## Step 2: Understanding ORPO 📚

### What is ORPO (Odds Ratio Preference Optimization)?

ORPO is a novel approach to aligning language models with human preferences, introduced in the paper [ORPO: Monolithic Preference Optimization without Reference Model](https://arxiv.org/abs/2403.07691).

### The Problem with Traditional Approaches

Traditional preference optimization (like RLHF or DPO) typically requires:
1. **Supervised Fine-Tuning (SFT)** - First train the model on good examples
2. **Preference Optimization** - Then align it using a reference model

This two-stage process is:
- Memory intensive (need to keep a reference model)
- Time consuming (two separate training phases)
- Complex to implement correctly

### How ORPO Solves This

ORPO combines both stages into **a single training phase** by:

1. **SFT Loss**: Encourages the model to generate high-quality "chosen" responses
2. **Odds Ratio Loss**: Penalizes generating "rejected" responses relative to chosen ones

The key insight is using the **odds ratio** (how much more likely is the chosen response vs rejected) as a penalty term, eliminating the need for a frozen reference model.

### Mathematical Intuition

```
L_ORPO = L_SFT + β × L_OR

Where:
- L_SFT = Standard cross-entropy loss on chosen responses
- L_OR = log(odds(chosen) / odds(rejected))  
- β = Weighting factor (typically 0.1)
```

This means: "Learn to generate good responses, AND make sure you prefer them over bad ones."

## Step 3: Load the Base Model 🦙

We'll use **Llama 3.2 1B Instruct** as our base model. This is a compact yet capable model that:

- Fits in 8GB VRAM with 4-bit quantization
- Already has instruction-following capabilities
- Is optimized by Unsloth for faster training

### Key Configuration Choices

| Parameter | Value | Why |
|-----------|-------|-----|
| `max_seq_length` | 1024 | Balance between context and memory usage |
| `dtype` | float16 | Standard precision for training |
| `load_in_4bit` | True | Reduces memory ~4x (essential for 8GB GPUs) |

> 💡 **4-bit Quantization**: Instead of storing each weight as a 32-bit float, we use only 4 bits. This reduces memory by 8x with minimal quality loss thanks to QLoRA techniques.

In [ ]:
# Step 3: Load the Base Model
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 1024  # Maximum tokens per training example

# Load the model with 4-bit quantization for memory efficiency
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",  # Unsloth's optimized version
    max_seq_length=max_seq_length,
    dtype=torch.float16,  # Use FP16 for training
    load_in_4bit=True,    # Enable 4-bit quantization (QLoRA)
)

# Set padding token (required for batch training)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Enable training mode with Unsloth optimizations
FastLanguageModel.for_training(model)

print(f"✓ Model loaded successfully!")
print(f"  - Max sequence length: {max_seq_length}")
print(f"  - Quantization: 4-bit")
print(f"  - Ready for ORPO training")

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Modelo cargado | max_seq=1024 | GGUF | listo para ORPO


## Step 4: Prepare the Training Dataset 📊

### About UltraFeedback

We'll use the **UltraFeedback** dataset, which contains:
- **Prompts**: User questions/instructions
- **Chosen responses**: High-quality responses (preferred by humans/GPT-4)
- **Rejected responses**: Lower-quality responses

This "binarized" format is perfect for preference optimization methods like ORPO.

### Data Processing Strategy

1. **Shuffle & Sample**: We use 5,000 examples (enough for meaningful learning, fits in memory)
2. **Format Conversion**: Transform chat format to ORPO's expected `{prompt, chosen, rejected}` structure
3. **Length Filtering**: Remove examples that would exceed our token limit (prevents OOM errors)

> ⚠️ **Why filter by length?** Each training example includes the prompt PLUS both responses. Long examples can cause GPU memory to overflow during the ORPO loss calculation.

In [ ]:
# Step 4: Load and Prepare the Dataset
from datasets import load_dataset

# Load UltraFeedback dataset (already has chosen/rejected pairs)
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

# Shuffle and select a subset for memory efficiency
dataset = dataset.shuffle(seed=42).select(range(5000))

def format_for_orpo(example):
    """
    Convert chat format to ORPO's expected format.
    
    Input format (chat):
        chosen: [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
        rejected: [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
    
    Output format (ORPO):
        prompt: "user's question"
        chosen: "good assistant response"
        rejected: "bad assistant response"
    """
    prompt = example["chosen"][0]["content"]           # User's message
    chosen_response = example["chosen"][1]["content"]  # Good response
    rejected_response = example["rejected"][1]["content"]  # Bad response
    return {"prompt": prompt, "chosen": chosen_response, "rejected": rejected_response}

# Apply formatting
train_dataset = dataset.map(
    format_for_orpo, 
    remove_columns=dataset.column_names,  # Remove original columns
    num_proc=1  # Single process (avoids Windows multiprocessing issues)
)

def filter_length(example):
    """Filter out examples that are too long (would cause OOM)."""
    total_chars = len(example["prompt"]) + len(example["chosen"]) + len(example["rejected"])
    return total_chars < 8000  # ~2000 tokens approximately

# Apply length filter
train_dataset = train_dataset.filter(filter_length, num_proc=1)

print(f"✓ Dataset prepared: {len(train_dataset)} examples")
print(f"\n📋 Sample example:")
print(f"  Prompt: {train_dataset[0]['prompt'][:100]}...")
print(f"  Chosen: {train_dataset[0]['chosen'][:100]}...")
print(f"  Rejected: {train_dataset[0]['rejected'][:100]}...")

README.md:   0%|          | 0.00/643 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/131M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62135 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter (num_proc=1):   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset listo: 4875 ejemplos (tras filtrar longitud)
Ejemplo: {'chosen': 'Reach out to friends, family, or significant others when feeling lonely at night...', 'rejected': 'The article provides suggestions for ways to combat loneliness at night. It enco...', 'prompt': 'INPUT ARTICLE: Article: Different laptop models have different numeric pad confi...'}


## Step 5: Configure LoRA and ORPO Training ⚙️

### What is LoRA (Low-Rank Adaptation)?

Instead of fine-tuning ALL model weights (billions of parameters), LoRA:
1. **Freezes** the original model weights
2. **Adds** small trainable matrices (adapters) to specific layers
3. **Trains** only these adapters (typically <1% of total parameters)

This dramatically reduces:
- Memory usage (can fine-tune on consumer GPUs)
- Training time (fewer parameters to update)
- Storage (save only the small adapter, not the full model)

### LoRA Configuration Explained

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| `r=16` | Rank of adapter matrices | Higher = more capacity, more memory |
| `lora_alpha=32` | Scaling factor (usually 2×r) | Controls learning rate scaling |
| `use_rslora=True` | Rank-stabilized LoRA | Better convergence with small datasets |
| `target_modules` | Attention + MLP layers | Which layers get adapters |

### ORPO Configuration Explained

| Parameter | Value | Explanation |
|-----------|-------|-------------|
| `beta=0.1` | Odds ratio weight | From the ORPO paper |
| `learning_rate=5e-6` | Very low LR | Stability with preference optimization |
| `per_device_train_batch_size=2` | Batch size | Limited by VRAM |
| `gradient_accumulation_steps=4` | Accumulation | Effective batch = 2×4 = 8 |
| `max_steps=200` | Training steps | ~8 passes through 5k examples |

In [ ]:
# Step 5: Configure LoRA Adapters and ORPO Training
from trl import ORPOTrainer, ORPOConfig

# === LoRA Configuration ===
# Add small trainable adapters to the frozen model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # Rank of LoRA matrices (capacity vs memory trade-off)
    lora_alpha=32,           # Scaling factor (typically 2*r)
    lora_dropout=0,          # No dropout (Unsloth handles regularization)
    target_modules=[         # Which layers to add adapters to
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj"       # MLP layers
    ],
    use_gradient_checkpointing="unsloth",  # Memory optimization
    use_rslora=True,         # Rank-stabilized LoRA (better for small datasets)
)

# === ORPO Training Configuration ===
orpo_config = ORPOConfig(
    output_dir="./llama1b_orpo_output",     # Where to save checkpoints
    
    # Core ORPO hyperparameters
    learning_rate=5e-6,                      # Low LR for stable preference learning
    beta=0.1,                                # Odds ratio weight (from paper)
    
    # Batch size configuration (effective batch = 2 * 4 = 8)
    per_device_train_batch_size=2,           # Limited by VRAM
    gradient_accumulation_steps=4,           # Accumulate gradients
    
    # Training duration
    max_steps=200,                           # Total training steps
    warmup_ratio=0.1,                        # 10% warmup
    logging_steps=10,                        # Log every 10 steps
    
    # Precision (auto-detect best for your GPU)
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Sequence length limits
    max_length=max_seq_length,               # Maximum total length
    max_prompt_length=512,                   # Truncate long prompts
    
    # Optimizer
    optim="adamw_8bit",                      # Memory-efficient optimizer
    
    # Required settings
    remove_unused_columns=False,             # ORPO needs all columns
    dataset_num_proc=1,                      # Avoid Windows multiprocessing issues
)

# === Create the Trainer ===
orpo_trainer = ORPOTrainer(
    model=model,
    args=orpo_config,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

print("✓ Training configuration complete!")
print(f"  - LoRA rank: 16")
print(f"  - Effective batch size: 8")
print(f"  - Total steps: 200")
print(f"\n🚀 Starting training...")

# Start training
orpo_trainer.train()

print("\n✓ Training complete!")

Unsloth 2026.1.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.
/content/unsloth_compiled_cache/UnslothORPOTrainer.py:727: UserWarning: This trainer will soon be moved to trl.experimental and is a candidate for removal. If you rely on it and want it to remain, please share your comments here: https://github.com/huggingface/trl/issues/4223. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  warnings.warn(


Map (num_proc=1):   0%|          | 0/4875 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/4875 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/4875 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,875 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / rejected,logps / chosen,logits / rejected,logits / chosen,log_odds_ratio,log_odds_chosen,eval_logits / chosen,eval_logits / rejected,nll_loss
10,1.838700,-0.174533,-0.158616,0.425000,-0.015918,-1.586158,-1.745334,-0.020704,0.125238,-0.967951,-0.168398,0,0,1.741945
20,2.035100,-0.175925,-0.183164,0.575000,0.007238,-1.831638,-1.759255,-0.255641,-0.012779,-0.791239,0.121413,No Log,No Log,1.955977
30,1.817500,-0.166657,-0.172318,0.525000,0.005660,-1.723179,-1.666574,-0.387127,0.118934,-0.822444,0.084131,No Log,No Log,1.735244
40,1.857900,-0.164119,-0.174677,0.537500,0.010558,-1.746774,-1.641191,-0.115981,0.135379,-0.779913,0.132143,No Log,No Log,1.779906
50,1.809800,-0.142907,-0.179132,0.612500,0.036226,-1.791324,-1.429067,0.000696,0.434524,-0.703702,0.408642,No Log,No Log,1.739451
60,1.883300,-0.158876,-0.160703,0.550000,0.001827,-1.607030,-1.588760,-0.177973,0.380030,-0.805272,0.050302,No Log,No Log,1.802793
70,1.915400,-0.164705,-0.157184,0.512500,-0.007521,-1.571836,-1.647046,0.206324,0.049902,-0.884313,-0.102785,No Log,No Log,1.826978
80,1.961100,-0.152390,-0.165627,0.562500,0.013237,-1.656271,-1.523898,0.115139,0.613229,-0.743090,0.162869,No Log,No Log,1.886783
90,2.015800,-0.170942,-0.162691,0.387500,-0.008251,-1.626909,-1.709418,0.158973,0.364949,-0.898978,-0.077301,No Log,No Log,1.925884
100,1.850500,-0.147924,-0.167155,0.550000,0.019231,-1.671550,-1.479236,-0.185108,0.366951,-0.778633,0.200013,No Log,No Log,1.772597


TrainOutput(global_step=200, training_loss=1.8727682876586913, metrics={'train_runtime': 970.6772, 'train_samples_per_second': 1.648, 'train_steps_per_second': 0.206, 'total_flos': 0.0, 'train_loss': 1.8727682876586913, 'epoch': 0.3281378178835111})

## Step 6: Export to GGUF Format 📦

### What is GGUF?

**GGUF** (GPT-Generated Unified Format) is a file format designed for efficient LLM inference:

- **Single file**: Contains model weights, tokenizer, and metadata
- **Quantization-friendly**: Supports various precision levels (F16, Q8, Q4, etc.)
- **Cross-platform**: Works with llama.cpp, Ollama, LM Studio, and more
- **Memory-mapped**: Can load parts of the model on demand

### Quantization Options

| Format | Size | Quality | Speed | Use Case |
|--------|------|---------|-------|----------|
| `f16` | Largest | Best | Slowest | Maximum quality, enough RAM |
| `q8_0` | ~50% | Excellent | Good | Good balance |
| `q4_k_m` | ~25% | Good | Fastest | Limited RAM/VRAM |

We'll export as **F16** (full precision) since we can always quantize further later.

In [ ]:
# Step 6: Export the Fine-Tuned Model to GGUF Format

save_path = "llama1b-orpo-gguf-f16"

# Unsloth provides a convenient method to export directly to GGUF
# This merges the LoRA adapters with the base model and converts to GGUF
model.save_pretrained_gguf(
    save_path,
    tokenizer,
    quantization_method="f16"  # Options: "f16", "q8_0", "q4_k_m", etc.
)

print(f"✓ Model exported to: {save_path}/")
print("  The GGUF file is ready for use with Ollama!")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [02:34<00:00, 154.84s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:34<00:00, 94.99s/it]


Unsloth: Merge process complete. Saved to `/content/llama1b-orpo-gguf-f16`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['llama-3.2-1b-instruct.F16.gguf']

{'save_directory': 'llama1b-orpo-gguf-f16',
 'gguf_files': ['llama-3.2-1b-instruct.F16.gguf'],
 'modelfile_location': '/content/Modelfile',
 'want_full_precision': True,
 'is_vlm': False,
 'fix_bos_token': False}

## Step 7: Verify the Export 🔍

Let's verify that our GGUF file was created successfully and check its size.

In [ ]:
# Step 7: Verify the Export
# List the output directory and find the GGUF file

!ls -lah /content
!ls -lah /content/llama1b-orpo-gguf-f16
!find /content -maxdepth 3 -name "*.gguf" -type f -print

print("\n✓ If you see a .gguf file above, the export was successful!")

total 2.4G
drwxr-xr-x  1 root root 4.0K Feb  1 21:01 .
drwxr-xr-x  1 root root 4.0K Feb  1 20:02 ..
drwxr-xr-x  4 root root 4.0K Dec  9 14:41 .config
drwxr-xr-x  4 root root 4.0K Feb  1 20:20 huggingface_tokenizers_cache
drwxr-xr-x  3 root root 4.0K Feb  1 20:41 llama1b-orpo-gguf
drwxr-xr-x  3 root root 4.0K Feb  1 20:45 llama1b-orpo-gguf-f16
-rw-r--r--  1 root root 2.4G Feb  1 21:01 llama-3.2-1b-instruct.F16.gguf
drwxr-xr-x  3 root root 4.0K Feb  1 20:37 llama4b_orpo_output
drwxr-xr-x 27 root root 4.0K Feb  1 20:58 llama.cpp
-rw-r--r--  1 root root 1.9K Feb  1 21:01 Modelfile
drwxr-xr-x  1 root root 4.0K Dec  9 14:42 sample_data
drwxr-xr-x  4 root root 4.0K Feb  1 20:17 unsloth_compiled_cache
total 2.4G
drwxr-xr-x 3 root root 4.0K Feb  1 20:45 .
drwxr-xr-x 1 root root 4.0K Feb  1 21:01 ..
drwxr-xr-x 3 root root 4.0K Feb  1 20:43 .cache
-rw-r--r-- 1 root root 3.8K Feb  1 20:43 chat_template.jinja
-rw-r--r-- 1 root root 1004 Feb  1 20:43 config.json
-rw-r--r-- 1 root root 2.4G Feb  1 20

## Step 8: Save to Google Drive (Optional) 💾

If you're running on Google Colab, you'll want to save your model to Google Drive before the session ends. This step:

1. Mounts your Google Drive
2. Creates an output directory
3. Copies both the **GGUF file** and **Modelfile** for Ollama

> 📁 After this step, you can download the files from your Drive and use them locally with Ollama.

In [ ]:
# Step 8: Save to Google Drive (for Colab users)
# This preserves your model after the Colab session ends

from google.colab import drive
import os
import glob
import shutil

# Mount Google Drive (you'll be prompted to authorize)
drive.mount("/content/drive")

# Create destination directory
dst_dir = "/content/drive/MyDrive/ollama_export_llama1b"
os.makedirs(dst_dir, exist_ok=True)

# Copy the Modelfile (tells Ollama how to use the model)
shutil.copy("/content/Modelfile", os.path.join(dst_dir, "Modelfile"))

# Find and copy the GGUF file (handles both possible locations)
candidates = [
    "/content/llama-3.2-1b-instruct.F16.gguf",
    *glob.glob("/content/llama1b-orpo-gguf-f16/*.gguf"),
    *glob.glob("/content/llama1b-orpo-gguf-f16/**/*.gguf", recursive=True),
]
ggufs = [p for p in candidates if os.path.isfile(p)]

if not ggufs:
    raise FileNotFoundError("No .gguf file found! Check the export step.")

gguf_path = ggufs[0]
shutil.copy(gguf_path, os.path.join(dst_dir, os.path.basename(gguf_path)))

print("✓ Files saved to Google Drive!")
print(f"  📁 Directory: {dst_dir}")
print(f"  📦 GGUF: {os.path.basename(gguf_path)}")
print(f"  📄 Modelfile: Modelfile")
print("\n👉 Download these files to use with Ollama locally!")

Mounted at /content/drive
Copiado a Drive en: /content/drive/MyDrive/ollama_export_llama1b
GGUF: /content/drive/MyDrive/ollama_export_llama1b/llama-3.2-1b-instruct.F16.gguf
Modelfile: /content/drive/MyDrive/ollama_export_llama1b/Modelfile


## Step 9: Test the Model 🧪

Before deploying, let's verify the model works by generating a response. This uses the model directly in Python (without Ollama) to confirm the fine-tuning was successful.

> 📝 **Note**: For production use, you'll deploy with Ollama (see the README for instructions).

In [ ]:
# Step 9: Test the Fine-Tuned Model
# Generate a response to verify everything works

# Test prompt - asking about ORPO (the technique we just used!)
prompt = "Explain what ORPO (Odds Ratio Preference Optimization) is in one sentence. Assume I don't understand anything about optimization."

# Format as chat message
messages = [{"role": "user", "content": prompt}]

# Apply the model's chat template
text = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

# Tokenize and move to GPU
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Generate response
outputs = model.generate(
    **inputs,
    max_new_tokens=128,   # Maximum length of response
    use_cache=True,       # Speed up generation
    do_sample=True,       # Enable sampling (not greedy)
    temperature=0.7       # Creativity level (0.0=deterministic, 1.0=creative)
)

# Decode and print the response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("🤖 Model Response:")
print("-" * 50)
print(response)
print("-" * 50)
print("\n✓ Model is working! Ready for deployment with Ollama.")

system

Cutting Knowledge Date: December 2023
Today Date: 01 Feb 2026

user

Explica qué es ORPO (Odds Ratio Preference Optimization) en una frase. Asume que no entiendo nada sobre optimización.assistant

No te preocupes, te explicaré de manera sencilla.

ORPO (Odds Ratio Preference Optimization) es una estrategia en el análisis de datos que se utiliza para encontrar el mejor modelo de clasificación que maximice la precisión y la precisión de la clasificación. En otras palabras, busca encontrar la combinación de variables que maximice la probabilidad de clasificar correctamente las categorías.


---

## 🎉 Congratulations!

You've successfully:

1. ✅ Loaded a pre-trained Llama 3.2 1B model
2. ✅ Fine-tuned it using ORPO preference optimization
3. ✅ Exported it to GGUF format for local deployment
4. ✅ Tested that it generates coherent responses

### What's Next?

**To deploy your model locally with Ollama:**

1. Download the `.gguf` file and `Modelfile` to your local machine
2. Install Ollama from [ollama.ai](https://ollama.ai)
3. Follow the instructions in the **README.md** to create and run your custom model

### Resources

- 📄 [ORPO Paper](https://arxiv.org/abs/2403.07691)
- 🦙 [Ollama Documentation](https://ollama.ai)
- ⚡ [Unsloth GitHub](https://github.com/unslothai/unsloth)
- 🤗 [TRL Library](https://github.com/huggingface/trl)